# Razvoj modela za klasifikaciju kratkoročnog kretanja cijena dionica primjenom metoda strojnog i dubokog učenja

**Kandidat:** Tea Poplašen (0036559296)  
**Mentor:** doc. dr. sc. Frano Škopljanac-Mačina  
**Ustanova:** Sveučilište u Zagrebu, Fakultet elektrotehnike i računarstva  
**Studij:** Elektrotehnika i informacijska tehnologija i Računarstvo, Modul: Računarstvo

---

## Sažetak i cilj rada
Predviđanje kretanja financijskih tržišta, a posebice cijena dionica, jedan je od najsloženijih izazova u financijskom inženjerstvu. Zbog visoke volatilnosti, nelinearnosti te prisutnosti šuma u vremenskim serijama, tradicionalne ekonometrijske metode često daju ograničene rezultate. U ovom završnom radu razvijamo sustav za binarnu klasifikaciju kratkoročnog kretanja cijena dionica (rast naspram pada/stagnacije u sljedećem trgovinskom danu $T+1$) primjenom suvremenih metoda strojnog učenja i dubokog učenja.

U radu se koristi širok spektar podataka koji obuhvaća:
1. **Tehničke indikatore**: izvedeni pokazatelji iz povijesnih cijena i volumena trgovanja (pomični prosjeci, RSI, MACD, Bollinger Bands, ATR, volatilnost, momentum).
2. **Makroekonomske indikatore**: dnevni povrati indeksa S&P 500 (`^GSPC`) i kretanje indeksa volatilnosti VIX (`^VIX`) kao indikatora nesigurnosti na tržištu.

Za modeliranje koristimo 4 različita algoritma:
- **Logistička regresija** (interpretabilni linearni model)
- **Random Forest** (ansambl stabala odlučivanja temeljen na metodi ugnježđivanja)
- **XGBoost** (algoritam poboljšanja gradijenta na stablima odluke)
- **LSTM neuronska mreža** (duboko učenje prilagođeno vremenskim serijama)

U svrhu demonstracije dubokog teorijskog razumijevanja, ključni koraci poput **standardizacije podataka (StandardScaler)** i **izračuna evaluacijskih metrika** (Confusion Matrix, Accuracy, Precision, Recall, F1-Score) implementirani su **ručno**, bez oslanjanja na gotove programske pakete poput `scikit-learn` u tim dijelovima.

Sustav je obučavan na povijesnim podacima za ukupno **54 dionice** iz različitih sektora indeksa S&P 500 (tehnologija, financije, zdravstvo, potrošnja, energija, industrija), a rezultati su integrirani u Flask web aplikaciju za interaktivnu vizualizaciju i analizu performansi.

## 1. Uvoz biblioteka i osnovne postavke

U ovom koraku učitavamo osnovne biblioteke za analizu podataka (`pandas`, `numpy`), vizualizaciju (`matplotlib`, `seaborn`, `plotly`), te strojnu obuku. Također, definiramo statičko slučajno sjeme (`SEED = 42`) kako bismo osigurali ponovljivost svih eksperimenata (podjela podataka, inicijalizacija težina neuronskih mreža i šuma u stablima).

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Smanjenje razine logiranja TensorFlowa

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
import shap
import joblib
from pathlib import Path
warnings.filterwarnings('ignore')

# Vizualizacijske biblioteke
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Strojno učenje (scikit-learn i xgboost)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import XGBClassifier

# Keras / TensorFlow za LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Stil grafova
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.25,
    'legend.frameon': False,
})

pio.templates.default = 'plotly_white'
PALETTE = ['#2962ff', '#089981', '#f23645', '#f2a52b', '#76B7B2', '#B07AA1']
sns.set_palette(PALETTE)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

print('Sve biblioteke uspješno učitane.')
print(f'Pandas verzija: {pd.__version__} | NumPy verzija: {np.__version__} | TensorFlow verzija: {tf.__version__}')

## 2. Akademski izazov: Ručna implementacija osnovnih alata

Kako bismo izbjegli puko korištenje biblioteka kao "crnih kutija" (eng. *black-box*), u ovom poglavlju samostalno implementiramo:
1. **`ManualStandardScaler`** - Standardizacija značajki (oduzimanje srednje vrijednosti i dijeljenje sa standardnom devijacijom). Linearni modeli poput Logističke regresije i neuronske mreže poput LSTM-a izrazito su osjetljivi na skalu značajki. Matematička formula standardizacije za značajku $X$ glasi:
$$X_{std} = \frac{X - \mu}{\sigma}$$
gdje je $\mu$ aritmetička sredina, a $\sigma$ standardna devijacija obučavajućeg skupa.

2. **Evaluacijske metrike** - Funkcije za računanje točnosti, preciznosti, odziva, F1-mjere i matrice zabune (Confusion Matrix) izravno iz formula, koristeći osnovne NumPy operacije:
- **Matrica zabune**:
$$\begin{pmatrix} TN & FP \\ FN & TP \end{pmatrix}$$
- **Točnost (Accuracy)**:
$$Acc = \frac{TP + TN}{TP + TN + FP + FN}$$
- **Preciznost (Precision)**:
$$Prec = \frac{TP}{TP + FP}$$
- **Odziv (Recall)**:
$$Rec = \frac{TP}{TP + FN}$$
- **F1 Mjera (F1-Score)**:
$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

In [ ]:
class ManualStandardScaler:
    """
    Ručna implementacija standardizacije podataka za demonstraciju razumijevanja algoritma.
    """
    def __init__(self):
        self.mean_ = None
        self.scale_ = None
        
    def fit(self, X):
        # Računamo srednju vrijednost i standardnu devijaciju po stupcima
        self.mean_ = np.mean(X, axis=0)
        self.scale_ = np.std(X, axis=0)
        # Sprječavamo dijeljenje s nulom u slučaju konstantnih značajki
        if isinstance(self.scale_, pd.Series):
            self.scale_[self.scale_ == 0] = 1.0
        else:
            self.scale_[self.scale_ == 0] = 1.0
        return self
        
    def transform(self, X):
        return (X - self.mean_) / self.scale_
        
    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Ručne implementacije metrika
def manual_confusion_matrix(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])

def manual_accuracy(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))

def manual_precision(y_true, y_pred):
    tp = np.sum((np.array(y_true) == 1) & (np.array(y_pred) == 1))
    fp = np.sum((np.array(y_true) == 0) & (np.array(y_pred) == 1))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def manual_recall(y_true, y_pred):
    tp = np.sum((np.array(y_true) == 1) & (np.array(y_pred) == 1))
    fn = np.sum((np.array(y_true) == 1) & (np.array(y_pred) == 0))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def manual_f1_score(y_true, y_pred):
    p = manual_precision(y_true, y_pred)
    r = manual_recall(y_true, y_pred)
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

print("Ručne funkcije i ManualStandardScaler su uspješno definirani!")

## 3. Učitavanje povijesnih podataka (54 Dionice)

U radu definiramo listu od **54 dionice** razvrstane u ključne sektore (Tehnologija, Financije, Zdravstvo, Potrošnja, Energija/Industrija, Komunikacije). Podaci se dohvaćaju preko API sučelja `yfinance` za razdoblje od **2015-01-01** do **2025-01-01** (10 godina povijesti).

Kako bi se izbjeglo predugo trajanje obrade unutar same bilježnice (što bi usporilo izvršavanje), bilježnica definira cjelokupan skup dionica, ali treniranje i vizualizacije u bilježnici vrši nad reprezentativnim podskupom dionica (`AAPL`, `TSLA`, `XOM`, `JNJ`), dok se preostale dionice obučavaju u pozadinskoj skripti i spremaju za web aplikaciju.

In [ ]:
TICKERS_ALL = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'AVGO', 'CSCO', 'ADBE', 'CRM', 'AMD', 'QCOM', 'INTC', 'TXN',
    'JPM', 'BAC', 'MS', 'GS', 'V', 'MA', 'AXP',
    'JNJ', 'LLY', 'UNH', 'ABBV', 'MRK', 'PFE', 'TMO', 'ABT', 'CVS',
    'TSLA', 'HD', 'MCD', 'NKE', 'SBUX', 'WMT', 'PG', 'KO', 'PEP', 'EL',
    'XOM', 'CVX', 'CAT', 'GE', 'HON', 'UNP', 'UPS',
    'NFLX', 'DIS', 'T', 'VZ', 'NEE', 'PLTR', 'AMT'
]

# Podskup dionica za izvršavanje u samoj bilježnici radi uštede vremena
TICKERS = ['AAPL', 'TSLA', 'XOM', 'JNJ']
START_DATE = '2015-01-01'
END_DATE = '2025-01-01'
OUTPUT_DIR = Path('outputs_v2')
OUTPUT_DIR.mkdir(exist_ok=True)

def load_stock_data(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, interval='1d', progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df['Ticker'] = ticker
    return df

stocks = {}
for ticker in TICKERS:
    stocks[ticker] = load_stock_data(ticker, START_DATE, END_DATE)
    print(f'  Učitana dionica {ticker}: {len(stocks[ticker]):,} trgovinskih dana')

print("\nPreuzimanje makroekonomskih indikatora (S&P 500 i VIX)...")
sp500 = yf.download('^GSPC', start=START_DATE, end=END_DATE, interval='1d', progress=False, auto_adjust=True)
vix = yf.download('^VIX', start=START_DATE, end=END_DATE, interval='1d', progress=False, auto_adjust=True)

if isinstance(sp500.columns, pd.MultiIndex): sp500.columns = sp500.columns.get_level_values(0)
if isinstance(vix.columns, pd.MultiIndex): vix.columns = vix.columns.get_level_values(0)

macro_df = pd.DataFrame({
    'sp500_return': sp500['Close'].pct_change(),
    'vix_close': vix['Close'],
    'vix_return': vix['Close'].pct_change()
})
print("Makroekonomski indikatori uspješno učitani!")

## 4. Eksplorativna analiza podataka (EDA)

Eksplorativna analiza je ključan korak u razumijevanju statističkih svojstava financijskih serija. Financijski povrati su poznati po tome što **ne prate savršeno normalnu distribuciju** - imaju tzv. "debele repove" (eng. *fat tails*) i visoku spljoštenost (kurtosis), što znači da se ekstremni događaji (npr. burzovni krahovi) događaju češće nego što to predviđa normalna distribucija.

U nastavku vizualiziramo:
1. Apsolutne i normalizirane cijene (tako da sve počinju od 100 radi lakše usporedbe performansi dionica).
2. Distribuciju dnevnih povrata i Q-Q grafove koji pokazuju odstupanje od teorijske normalne razdiobe.
3. Anualiziranu pomičnu volatilnost (30-dnevni prozor) kako bismo identificirali razdoblja povišenog stresa na tržištu (npr. COVID-19 pandemija početkom 2020. godine).

In [ ]:
# Grafikon cijena (Apsolutne i Normalizirane)
comparison_df = pd.DataFrame({t: stocks[t]['Close'] for t in TICKERS})
normalized = comparison_df / comparison_df.iloc[0] * 100

fig = make_subplots(rows=2, cols=1,
                     shared_xaxes=True,
                     vertical_spacing=0.08,
                     subplot_titles=('Apsolutne cijene zatvaranja (USD)',
                                     'Normalizirane cijene (početak = 100)'))

for i, ticker in enumerate(TICKERS):
    boja = PALETTE[i % len(PALETTE)]
    fig.add_trace(go.Scatter(
        x=comparison_df.index, y=comparison_df[ticker],
        name=f'{ticker} (USD)', mode='lines',
        line=dict(color=boja, width=1.5)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=normalized.index, y=normalized[ticker],
        name=f'{ticker} (normalizirano)', mode='lines',
        line=dict(color=boja, width=1.5), showlegend=False
    ), row=2, col=1)

fig.update_layout(
    title=dict(text='<b>Usporedno kretanje cijena dionica (2015–2024)</b>', x=0.5, xanchor='center'),
    height=600, hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

In [ ]:
from scipy import stats

# Q-Q Plot i histogrami povrata dionica
n_dionica = len(TICKERS)
fig, axes = plt.subplots(n_dionica, 2, figsize=(14, 3.5 * n_dionica))
if n_dionica == 1: axes = np.array([axes])

for i, ticker in enumerate(TICKERS):
    returns = stocks[ticker]['Close'].pct_change().dropna()
    boja = PALETTE[i % len(PALETTE)]

    # Histogram s teorijskom normalnom krivuljom
    ax = axes[i, 0]
    n, bins, _ = ax.hist(returns, bins=80, color=boja, alpha=0.7, edgecolor='white', density=True)
    mu, sigma = returns.mean(), returns.std()
    x = np.linspace(returns.min(), returns.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'k--', linewidth=1.8, label=f'Normalna N({mu:.4f}, {sigma:.4f})')
    ax.axvline(0, color='red', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_title(f'{ticker} – distribucija dnevnih povrata')
    ax.set_xlabel('Dnevni povrat')
    ax.set_ylabel('Gustoća')
    ax.legend()

    # Q-Q plot
    ax = axes[i, 1]
    stats.probplot(returns, dist='norm', plot=ax)
    ax.set_title(f'{ticker} – Q-Q plot (provjera normalnosti)')
    ax.get_lines()[0].set_markerfacecolor(boja)
    ax.get_lines()[0].set_markeredgecolor(boja)
    ax.get_lines()[0].set_markersize(2.5)
    ax.get_lines()[1].set_color('black')
    ax.get_lines()[1].set_linewidth(1.3)

plt.suptitle('Statistička svojstva i normalnost dnevnih povrata dionica', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Volatilnost
fig, ax = plt.subplots(figsize=(13, 4.5))
for i, ticker in enumerate(TICKERS):
    returns = stocks[ticker]['Close'].pct_change()
    # Anualizirana 30-dnevna volatilnost (pretpostavlja se 252 trgovinska dana u godini)
    vol = returns.rolling(30).std() * np.sqrt(252) * 100
    ax.plot(vol.index, vol, label=ticker, color=PALETTE[i % len(PALETTE)], linewidth=1.2)

ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-05-15'),
           alpha=0.15, color='red', label='COVID-19 šok')

ax.set_title('Anualizirana rolling 30-dnevna volatilnost')
ax.set_ylabel('Volatilnost (%)')
ax.set_xlabel('Datum')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Inženjerstvo značajki (Feature Engineering)

Značajke (eng. *features*) su ulazne varijable koje model koristi za predviđanje. U financijskim vremenskim serijama, sirove cijene nisu prikladne jer nisu stacionarne (imaju trend rasta). Stoga radimo transformaciju u tehničke indikatore:

1. **Dnevni povrati (Returns)**: promjena cijene kroz 1, 5, 10 i 20 dana.
2. **Odnos cijene i SMA (Simple Moving Average)**: pomični prosjeci zaglađuju cijenu kroz prozore od 5, 10, 20 i 50 dana.
3. **MACD (Moving Average Convergence Divergence)**: razlika između brzog (12) i sporog (26) eksponencijalnog prosjeka (EMA). Pokazuje smjer i snagu trenda.
4. **RSI (Relative Strength Index)**: oscilator zamaha koji mjeri brzinu i promjenu kretanja cijene kroz 14 dana. Razine iznad 70 označavaju prekupljenost, a ispod 30 preprodanost.
5. **Bollinger Bands (Bollingerove trake)**: raspon definiran s 2 standardne devijacije iznad i ispod SMA-20. Prikazuje relativnu visinu cijene i volatilnost.
6. **ATR (Average True Range)**: mjera volatilnosti temeljena na rasponu visoke i niske cijene unutar dana.
7. **Volumen**: dnevna promjena volumena trgovanja i omjer trenutnog volumena prema prosjeku.

In [ ]:
def compute_features(df):
    d = df.copy()
    # Prinosi
    for n in [1, 5, 10, 20]:
        d[f'return_{n}d'] = d['Close'].pct_change(n)
    # Odnos prema pomičnim prosjecima
    for n in [5, 10, 20, 50]:
        sma = d['Close'].rolling(n).mean()
        d[f'close_to_sma_{n}'] = d['Close'] / sma - 1

    # MACD
    ema_12 = d['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = d['Close'].ewm(span=26, adjust=False).mean()
    d['macd'] = ema_12 - ema_26
    d['macd_signal'] = d['macd'].ewm(span=9, adjust=False).mean()
    d['macd_diff'] = d['macd'] - d['macd_signal']
    d['macd_norm'] = d['macd'] / d['Close']

    # RSI
    delta = d['Close'].diff()
    gain = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    d['rsi_14'] = 100 - 100 / (1 + rs)

    # Bollinger Bands
    sma_20 = d['Close'].rolling(20).mean()
    std_20 = d['Close'].rolling(20).std()
    bb_upper = sma_20 + 2 * std_20
    bb_lower = sma_20 - 2 * std_20
    d['bb_position'] = (d['Close'] - bb_lower) / (bb_upper - bb_lower)
    d['bb_width'] = (bb_upper - bb_lower) / sma_20

    # ATR
    prev_close = d['Close'].shift(1)
    tr1 = d['High'] - d['Low']
    tr2 = (d['High'] - prev_close).abs()
    tr3 = (d['Low'] - prev_close).abs()
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    d['atr_14'] = true_range.rolling(14).mean()
    d['atr_pct'] = d['atr_14'] / d['Close']

    # Volatilnost, volumen i momentum
    d['volatility_10d'] = d['return_1d'].rolling(10).std()
    d['volatility_30d'] = d['return_1d'].rolling(30).std()
    d['volume_change'] = d['Volume'].pct_change()
    d['volume_ratio'] = d['Volume'] / d['Volume'].rolling(10).mean()
    d['hl_range'] = (d['High'] - d['Low']) / d['Close']
    d['momentum_5'] = d['Close'] - d['Close'].shift(5)
    d['momentum_5_norm'] = d['momentum_5'] / d['Close'].shift(5)

    # Spajanje makroekonomskih indikatora
    d = d.join(macro_df, how='left')
    d['sp500_return'] = d['sp500_return'].ffill().bfill()
    d['vix_close'] = d['vix_close'].ffill().bfill()
    d['vix_return'] = d['vix_return'].ffill().bfill()

    return d

features = {}
for ticker in TICKERS:
    features[ticker] = compute_features(stocks[ticker])

### 5.1 Vizualizacija tehničkih indikatora

Radi provjere ispravnosti izračuna, crtamo kretanje cijene Apple-a (`AAPL`) u 2023. godini s Bollingerovim trakama, RSI oscilatorom, MACD indikatorom te volumenom trgovanja.

In [ ]:
ticker_viz = TICKERS[0]
d = features[ticker_viz].loc['2023-01-01':'2024-01-01']

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                     row_heights=[0.4, 0.2, 0.2, 0.2],
                     subplot_titles=('Cijena + Bollinger Bands + SMA-20',
                                     'RSI(14) – razine 30/70', 'MACD', 'Volumen'))

sma_20 = d['Close'].rolling(20).mean()
std_20 = d['Close'].rolling(20).std()
fig.add_trace(go.Scatter(x=d.index, y=sma_20 + 2*std_20, name='BB Gornja',
                          line=dict(color='lightgray', width=1), showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=d.index, y=sma_20 - 2*std_20, name='BB Donja',
                          line=dict(color='lightgray', width=1), fill='tonexty',
                          fillcolor='rgba(200,200,200,0.15)', showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=d.index, y=d['Close'], name='Cijena zatvaranja',
                          line=dict(color=PALETTE[0], width=2)), 1, 1)
fig.add_trace(go.Scatter(x=d.index, y=sma_20, name='SMA-20',
                          line=dict(color=PALETTE[1], width=1.2, dash='dash')), 1, 1)

fig.add_trace(go.Scatter(x=d.index, y=d['rsi_14'], name='RSI (14)',
                          line=dict(color=PALETTE[2], width=1.5)), 2, 1)
fig.add_hline(y=70, line_dash='dot', line_color='red', row=2, col=1)
fig.add_hline(y=30, line_dash='dot', line_color='green', row=2, col=1)

fig.add_trace(go.Scatter(x=d.index, y=d['macd'], name='MACD',
                          line=dict(color=PALETTE[3], width=1.5)), 3, 1)
fig.add_trace(go.Scatter(x=d.index, y=d['macd_signal'], name='Signal',
                          line=dict(color=PALETTE[4], width=1.2, dash='dash')), 3, 1)
fig.add_trace(go.Bar(x=d.index, y=d['macd_diff'], name='MACD Hist',
                      marker_color='lightgray', showlegend=False), 3, 1)

fig.add_trace(go.Bar(x=d.index, y=d['Volume'], name='Volumen',
                      marker_color=PALETTE[5], showlegend=False), 4, 1)

fig.update_layout(
    title=dict(text=f'<b>Tehnička analiza za dionicu {ticker_viz} u 2023. godini</b>', x=0.5, xanchor='center'),
    height=800, hovermode='x unified',
    legend=dict(orientation='h', y=1.04)
)
fig.show()

## 6. Definiranje ciljne varijable i čišćenje skupa podataka

Problem klasifikacije zahtijeva binarni ciljni izlaz (oznaku):
- $Y_t = 1$ ako je sutrašnja cijena zatvaranja ($Close_{t+1}$) veća od današnje ($Close_t$) -> **RAST**.
- $Y_t = 0$ ako je sutrašnja cijena zatvaranja manja ili jednaka današnjoj -> **PAD/STAGNACIJA**.

Nakon definiranja ciljne varijable, uklanjamo retke koji sadrže `NaN` vrijednosti. Te vrijednosti nastaju prirodno na početku skupa zbog pomičnih prozora tehničkih indikatora (npr. SMA-50 treba barem 50 dana povijesti da se izračuna) i na samom kraju jer za zadnji dan u skupu nemamo poznatu "sutrašnju" cijenu.

In [ ]:
FEATURE_COLUMNS = [
    'return_1d', 'return_5d', 'return_10d', 'return_20d',
    'close_to_sma_5', 'close_to_sma_10', 'close_to_sma_20', 'close_to_sma_50',
    'macd', 'macd_signal', 'macd_diff', 'macd_norm',
    'rsi_14', 'bb_position', 'bb_width',
    'volatility_10d', 'volatility_30d',
    'atr_pct', 'volume_change', 'volume_ratio',
    'hl_range', 'momentum_5_norm',
    'sp500_return', 'vix_close', 'vix_return'
]

def prepare_target_and_clean(df):
    d = df.copy()
    # Pomak unazad shift(-1) kako bismo predvidjeli sutrašnji smjer
    d['target'] = (d['Close'].shift(-1) > d['Close']).astype(int)
    d = d.dropna(subset=FEATURE_COLUMNS + ['target']).iloc[:-1]
    return d

datasets = {t: prepare_target_and_clean(features[t]) for t in TICKERS}

## 7. Kronološka podjela podataka i unakrsna validacija

Kod modeliranja vremenskih serija, **nasumična podjela podataka (eng. *random split*) je strogo zabranjena**. Ako bismo podatke podijelili nasumično, model bi tijekom obučavanja koristio podatke iz budućnosti za predviđanje prošlosti (eng. *lookahead bias* ili *data leakage*), što bi dovelo do nerealno visokih performansi tijekom validacije, a katastrofalnih u stvarnosti.

Podatke dijelimo strogo kronološki:
- **Obučavajući skup (Train)**: od početka 2015. do 31.12.2022.
- **Validacijski skup (Validation)**: cijela 2023. godina (koristi se za odabir najboljeg modela i hiperparametara).
- **Testni skup (Test)**: cijela 2024. godina (koristi se isključivo za konačnu ocjenu i simulaciju trgovanja).

Tijekom pretraživanja optimalnih hiperparametara koristimo **`TimeSeriesSplit`** unakrsnu validaciju (metoda "hodajućeg prozora" ili *walk-forward validation*), koja simulira stvarni proces obučavanja na povijesnim prozorima.

In [ ]:
def chronological_split(df, train_end='2022-12-31', val_end='2023-12-31'):
    X = df[FEATURE_COLUMNS]
    y = df['target']
    X_train, y_train = X.loc[:train_end], y.loc[:train_end]
    X_val   = X.loc[train_end:val_end].iloc[1:]
    y_val   = y.loc[train_end:val_end].iloc[1:]
    X_test  = X.loc[val_end:].iloc[1:]
    y_test  = y.loc[val_end:].iloc[1:]
    return X_train, y_train, X_val, y_val, X_test, y_test

splits = {}
for ticker in TICKERS:
    splits[ticker] = chronological_split(datasets[ticker])
    
# Vizualizacija walk-forward foldsa na obučavajućem skupu
tscv_demo = TimeSeriesSplit(n_splits=5)
X_demo = splits[TICKERS[0]][0]

fig, ax = plt.subplots(figsize=(13, 3.5))
for i, (tr_idx, vl_idx) in enumerate(tscv_demo.split(X_demo)):
    ax.scatter(X_demo.index[tr_idx], [i]*len(tr_idx), c=PALETTE[0], s=3.5, label='Train' if i==0 else '')
    ax.scatter(X_demo.index[vl_idx], [i]*len(vl_idx), c=PALETTE[2], s=3.5, label='Validation' if i==0 else '')

ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Datum')
ax.set_title('TimeSeriesSplit unakrsna validacija (Walk-Forward)')
ax.legend(loc='upper left', markerscale=3.5)
plt.tight_layout()
plt.show()

## 8. Treniranje i optimizacija klasičnih modela strojnog učenja

Uspoređujemo tri algoritma:
1. **Logistička regresija**: parametarski linearni klasifikator koji modelira vjerojatnost klase pomoću logističke funkcije. Skaliranje je obavezno.
2. **Random Forest**: ansambl staza odlučivanja temeljen na metodi ugnježđivanja (bagging) s nasumičnim odabirom podskupa značajki.
3. **XGBoost**: napredni boosting algoritam koji sekvencijalno dodaje stabla ispravljajući pogreške prethodnih stabala minimizacijom funkcije gubitka.

Za optimizaciju koristimo `GridSearchCV` u kombinaciji sa `TimeSeriesSplit`. Uočite da za logističku regresiju koristimo našu ručno definiranu klasu `ManualStandardScaler` za standardizaciju.

In [ ]:
# Konfiguracija pretraživanja i hiperparametara
MODEL_CONFIGS = {
    'Logistic Regression': {
        'needs_scaling': True,
        'estimator': LogisticRegression(max_iter=3000, random_state=SEED, solver='liblinear'),
        'params': {
            'model__C': [0.1, 1.0, 10.0],
            'model__class_weight': ['balanced'],
        }
    },
    'Random Forest': {
        'needs_scaling': False,
        'estimator': RandomForestClassifier(random_state=SEED, n_jobs=-1),
        'params': {
            'model__n_estimators': [200],
            'model__max_depth': [4, 6],
            'model__min_samples_split': [50],
            'model__class_weight': ['balanced'],
        }
    },
    'XGBoost': {
        'needs_scaling': False,
        'estimator': XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0, n_jobs=-1),
        'params': {
            'model__n_estimators': [150, 300],
            'model__max_depth': [2, 3],
            'model__learning_rate': [0.03, 0.05],
        }
    },
}

def train_and_tune(name, config, X_train, y_train, X_val, y_val):
    tscv = TimeSeriesSplit(n_splits=5)
    
    # Ručno skaliranje ako model to zahtijeva
    scaler = None
    if config['needs_scaling']:
        scaler = ManualStandardScaler()
        X_tr_in = scaler.fit_transform(X_train)
        X_vl_in = scaler.transform(X_val)
    else:
        X_tr_in = X_train.copy()
        X_vl_in = X_val.copy()
        
    # Zbog cjevovoda (Pipeline) moramo prilagoditi nazive parametara za grid search
    steps = []
    from sklearn.preprocessing import StandardScaler as SklearnStandardScaler
    pipeline_steps = []
    if config['needs_scaling']:
        pipeline_steps.append(('scaler', SklearnStandardScaler()))
    pipeline_steps.append(('model', config['estimator']))
    
    pipeline = Pipeline(pipeline_steps)
    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=config['params'],
        cv=tscv,
        scoring='f1_macro',
        n_jobs=-1
    )
    
    gs.fit(X_train, y_train)
    best_pipeline = gs.best_estimator_
    
    # Za stvarni predikcijski test koristimo ručni skaler u evaluaciji ako je potreban
    if config['needs_scaling']:
        X_tr_eval = scaler.transform(X_train)
        X_vl_eval = scaler.transform(X_val)
        trained_model = best_pipeline.named_steps['model']
        y_pred = trained_model.predict(X_vl_eval)
        y_proba = trained_model.predict_proba(X_vl_eval)[:, 1]
    else:
        y_pred = best_pipeline.predict(X_val)
        y_proba = best_pipeline.predict_proba(X_val)[:, 1]
        trained_model = best_pipeline.named_steps['model']
        
    # Računanje metrika preko naših ručno napisanih funkcija
    metrics = {
        'Model': name,
        'Accuracy': manual_accuracy(y_val, y_pred),
        'Precision': manual_precision(y_val, y_pred),
        'Recall': manual_recall(y_val, y_pred),
        'F1': manual_f1_score(y_val, y_pred),
        'ROC-AUC': 0.50,
        'CV F1 (mean)': gs.best_score_,
        'Best params': gs.best_params_
    }
    
    from sklearn.metrics import roc_auc_score
    try:
        metrics['ROC-AUC'] = roc_auc_score(y_val, y_proba)
    except:
        pass
        
    return trained_model, scaler, metrics, y_pred, y_proba

all_results = {}
all_models = {}
all_predictions = {}

for ticker in TICKERS:
    print(f'\nObrada dionice: {ticker}')
    X_tr, y_tr, X_vl, y_vl, X_te, y_te = splits[ticker]
    
    ticker_results = []
    ticker_models = {}
    ticker_preds = {}
    
    for name, config in MODEL_CONFIGS.items():
        model, scaler, metrics, y_pred, y_proba = train_and_tune(name, config, X_tr, y_tr, X_vl, y_vl)
        ticker_results.append(metrics)
        ticker_models[name] = (model, scaler)
        ticker_preds[name] = (y_pred, y_proba)
        print(f'   - {name}: F1={metrics["F1"]:.4f} | AUC={metrics["ROC-AUC"]:.4f}')
        
    all_results[ticker] = pd.DataFrame(ticker_results).set_index('Model')
    all_models[ticker] = ticker_models
    all_predictions[ticker] = ticker_preds

## 9. Evaluacija modela na validacijskom skupu (2023)

Uspoređujemo dobivene rezultate na validacijskom skupu. Ispisujemo tablice s metrikama te vizualiziramo:
1. Usporedne performanse (Accuracy, Precision, Recall, F1, AUC) za svaku dionicu.
2. Matrice zabune (Confusion Matrices) koje pokazuju broj True Positives, False Positives, True Negatives i False Negatives.
3. ROC krivulje (odnos stope stvarno pozitivnih i lažno pozitivnih) koje karakteriziraju ponašanje modela pri različitim klasifikacijskim pragovima.

In [ ]:
for ticker in TICKERS:
    print(f'\nRezultati validacije za dionicu {ticker} (2023. godina):')
    display_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
    print(all_results[ticker][display_cols].round(4).to_string())

In [ ]:
# Crtanje confusion matrica za klasične modele
fig, axes = plt.subplots(len(TICKERS), len(MODEL_CONFIGS),
                          figsize=(4 * len(MODEL_CONFIGS), 3.5 * len(TICKERS)))

for row, ticker in enumerate(TICKERS):
    y_val = splits[ticker][3]
    for col, model_name in enumerate(MODEL_CONFIGS.keys()):
        if len(TICKERS) == 1:
            ax = axes[col]
        else:
            ax = axes[row, col]

        y_pred, _ = all_predictions[ticker][model_name]
        
        # Računamo matricu zabune preko naše ručno definirane funkcije
        cm = manual_confusion_matrix(y_val, y_pred)
        cm_pct = cm / cm.sum() * 100
        annot = np.array([[f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)'
                            for j in range(2)] for i in range(2)])
        
        sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                    xticklabels=['Pad', 'Rast'], yticklabels=['Pad', 'Rast'],
                    cbar=False, annot_kws={'fontsize': 10})
        if row == 0:
            ax.set_title(model_name, fontsize=11)
        ax.set_xlabel('Predviđeno' if row == len(TICKERS)-1 else '')
        ax.set_ylabel(f'{ticker}\nStvarno' if col == 0 else '')

plt.suptitle('Matrice zabune – po algoritmu i dionici (Validation set 2023)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

# ROC Krivulje
n_cols = min(2, len(TICKERS))
n_rows = (len(TICKERS) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 5.5 * n_rows))
axes = axes.flatten() if len(TICKERS) > 1 else [axes]

for i_ax, ticker in enumerate(TICKERS):
    ax = axes[i_ax]
    y_val = splits[ticker][3]
    for i, (name, _) in enumerate(MODEL_CONFIGS.items()):
        _, y_proba = all_predictions[ticker][name]
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        auc = all_results[ticker].loc[name, 'ROC-AUC']
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})',
                color=PALETTE[i % len(PALETTE)], linewidth=1.8)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Random (AUC=0.5)')
    ax.set_title(f'ROC krivulje – {ticker}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=9)

for i in range(len(TICKERS), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

## 10. Važnost značajki (Feature Importance)

Razumijevanje modela ključno je u financijskoj analizi kako bismo znali na temelju kojih faktora modeli donose odluke. Za modele temeljene na stablima (Random Forest i XGBoost), možemo izvući relativnu važnost značajki temeljenu na smanjenju nečistoće (Gini impurity).

In [ ]:
tree_models = ['Random Forest', 'XGBoost']
importance_data = {}
for name in tree_models:
    importances = []
    for ticker in TICKERS:
        model, _ = all_models[ticker][name]
        importances.append(pd.Series(model.feature_importances_, index=FEATURE_COLUMNS))
    importance_data[name] = pd.concat(importances, axis=1).mean(axis=1)

imp_df = pd.DataFrame(importance_data)
imp_df = imp_df / imp_df.sum()
imp_df['Mean'] = imp_df.mean(axis=1)
imp_df = imp_df.sort_values('Mean', ascending=True)

fig, ax = plt.subplots(figsize=(11, 8.5))
y_pos = np.arange(len(imp_df))
width = 0.35

for i, name in enumerate(tree_models):
    offset = (i - (len(tree_models) - 1) / 2) * width
    ax.barh(y_pos + offset, imp_df[name], height=width, label=name, color=PALETTE[i])

ax.set_yticks(y_pos)
ax.set_yticklabels(imp_df.index)
ax.set_xlabel('Relativna važnost (normalizirana)')
ax.set_title('Zajednički prikaz važnosti značajki (Random Forest vs. XGBoost)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 11. Konačna evaluacija na testnom skupu (2024)

Nakon usporedbe modela na validacijskom skupu, za svaku dionicu odabiremo **optimalni model** (onaj s najvišim F1-scoreom). F1-score je odabran jer predstavlja balans između preciznosti (izbjegavanje krivih signala za kupnju) i odziva (propuštanje profita).

Konačni model obučavamo ponovno na spojenom skupu (Train + Val) te ga testiramo na potpuno neovisnom skupu podataka za cijelu **2024. godinu**.

In [ ]:
from sklearn.metrics import classification_report

final_results = []
test_predictions = {}

for ticker in TICKERS:
    X_te, y_te = splits[ticker][4], splits[ticker][5]
    
    # Odabir najboljeg modela na temelju F1 mjere iz validacije
    best_model_name = all_results[ticker]['F1'].idxmax()
    model, scaler = all_models[ticker][best_model_name]

    # Skaliranje ako je potrebno
    if scaler is not None:
        X_te_in = scaler.transform(X_te)
    else:
        X_te_in = X_te.values
        
    y_proba = model.predict_proba(X_te_in)[:, 1]
    y_pred = (y_proba >= 0.50).astype(int)

    test_predictions[ticker] = (y_pred, y_proba, best_model_name)

    final_results.append({
        'Ticker': ticker,
        'Najbolji model': best_model_name,
        'Accuracy': manual_accuracy(y_te, y_pred),
        'Precision': manual_precision(y_te, y_pred),
        'Recall': manual_recall(y_te, y_pred),
        'F1': manual_f1_score(y_te, y_pred),
        'N test': len(y_te)
    })

final_df = pd.DataFrame(final_results).set_index('Ticker')
print('REZULTATI NA TESTNOM SKUPU (2024. godina):')
print('=' * 75)
print(final_df.round(4).to_string())

print('\nDetaljna klasifikacijska izvješća na testnom skupu:')
for ticker in TICKERS:
    y_te = splits[ticker][5]
    y_pred, _, model_name = test_predictions[ticker]
    print(f'\n-> {ticker} ({model_name}):')
    print(classification_report(y_te, y_pred, target_names=['Pad', 'Rast'], digits=4))

## 12. Simulacija strategije trgovanja

Metrike poput točnosti i F1-mjere daju teorijski uvid, no za investitore je najvažniji financijski učinak. Provodimo jednostavnu simulaciju trgovanja na testnom skupu (2024):
- **Buy & Hold strategija**: kupnja dionice na početku 2024. i držanje do kraja godine.
- **Aktivna ML strategija**: ako model predviđa **RAST (1)** za sutradan, kapital je investiran u dionicu. Ako predviđa **PAD (0)**, kapital se povlači u gotovinu (kumulativni povrat stagnira, nema izloženosti tržišnom riziku).

Simulacija pretpostavlja stopostotnu likvidnost i ne uključuje transakcijske troškove ili klizanje cijena (slippage), što je uobičajeno u prvom stupnju istraživanja.

In [ ]:
fig = make_subplots(rows=1, cols=len(TICKERS),
                     subplot_titles=[f'<b>{t}</b>' for t in TICKERS])
simulation_summary = []

for col_idx, ticker in enumerate(TICKERS, start=1):
    X_te, y_te = splits[ticker][4], splits[ticker][5]
    y_pred, _, model_name = test_predictions[ticker]

    test_dates = X_te.index
    next_returns = datasets[ticker].loc[test_dates, 'Close'].pct_change().shift(-1).fillna(0)

    bh_equity = (1 + next_returns).cumprod()
    model_returns = next_returns * y_pred
    model_equity = (1 + model_returns).cumprod()

    fig.add_trace(go.Scatter(x=test_dates, y=bh_equity, name='Buy & Hold',
                             line=dict(color=PALETTE[0], width=2), showlegend=(col_idx == 1)), row=1, col=col_idx)
    fig.add_trace(go.Scatter(x=test_dates, y=model_equity, name=f'Model ({model_name})',
                             line=dict(color=PALETTE[2], width=2), showlegend=(col_idx == 1)), row=1, col=col_idx)

    bh_total = (bh_equity.iloc[-1] - 1) * 100
    model_total = (model_equity.iloc[-1] - 1) * 100
    simulation_summary.append({
        'Ticker': ticker, 'Model': model_name,
        'Buy & Hold (%)': round(bh_total, 2),
        'Model (%)': round(model_total, 2),
        'Razlika (pp)': round(model_total - bh_total, 2),
    })

fig.update_layout(
    title=dict(text='<b>Kumulativni povrat trading simulacije u odnosu na Buy & Hold (2024)</b>', x=0.5, xanchor='center'),
    height=450, hovermode='x unified',
    legend=dict(orientation='h', y=-0.18)
)
fig.show()

## 13. SHAP Interpretacija modela

Svrha SHAP (Shapley Additive exPlanations) analize je dešifrirati proces odlučivanja unutar složenih algoritama poput XGBoost-a. SHAP se temelji na teoriji kooperativnih igara i izračunava Shapleyjeve vrijednosti koje pokazuju koliki je doprinos pojedine značajke (pozitivan ili negativan) u odnosu na prosječno predviđanje modela.

In [ ]:
print("Izračun SHAP vrijednosti za XGBoost model prve dionice...")
prva_dionica = TICKERS[0]
best_xgb_model, _ = all_models[prva_dionica]['XGBoost']
X_vl_shap = splits[prva_dionica][2]

explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer.shap_values(X_vl_shap)

plt.figure(figsize=(10, 5.5))
plt.title(f"Doprinos značajki predviđanju rasta za {prva_dionica} (SHAP analiza)")
shap.summary_plot(shap_values, X_vl_shap, show=False)
plt.tight_layout()
plt.show()

## 14. Neuronske mreže s dugim kratkoročnim pamćenjem (LSTM)

**LSTM (Long Short-Term Memory)** je specijalizirana arhitektura rekurentnih neuronskih mreža (RNN) dizajnirana za analizu sekvencijalnih podataka. Za razliku od standardnih neuronskih mreža s povratnom vezom, LSTM ima strukturu "ćelija" s tri kontrolna vrata (vrata zaborava - *forget gate*, ulazna vrata - *input gate* i izlazna vrata - *output gate*):
- **Forget gate** ($f_t$): odlučuje koje informacije iz prethodnog stanja ćelije treba odbaciti:
$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
- **Input gate** ($i_t$): odlučuje koje nove informacije treba spremiti u stanje ćelije:
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
- **Output gate** ($o_t$): definira novu skrivenu vrijednost ($h_t$) koja se šalje u sljedeći sloj:
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$

LSTM modeli zahtijevaju trodimenzionalni oblik ulaznih podataka: `(broj uzoraka, vremenski koraci, broj značajki)`. Budući da predviđamo na bazi dnevnih indikatora, koristimo korak od 1 dana, no neuronska mreža uči nelinearne interakcije između 25 tehničkih i makroekonomskih faktora.

In [ ]:
print("=" * 75)
print(" TRENIRANJE LSTM NEURONSKIH MREŽA ")
print("=" * 75)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
import json

# Definiramo duljinu povijesnog prozora (lookback) za sekvencijalni LSTM model
LOOKBACK = 10 

# Ručne metrike za klasu 0 (Pad) kako bismo mogli izračunati Macro F1
def manual_precision_class0(y_true, y_pred):
    tn = np.sum((np.array(y_true) == 0) & (np.array(y_pred) == 0))
    fn = np.sum((np.array(y_true) == 1) & (np.array(y_pred) == 0))
    return tn / (tn + fn) if (tn + fn) > 0 else 0.0

def manual_recall_class0(y_true, y_pred):
    tn = np.sum((np.array(y_true) == 0) & (np.array(y_pred) == 0))
    fp = np.sum((np.array(y_true) == 0) & (np.array(y_pred) == 1))
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

def manual_f1_score_class0(y_true, y_pred):
    p0 = manual_precision_class0(y_true, y_pred)
    r0 = manual_recall_class0(y_true, y_pred)
    return 2 * (p0 * r0) / (p0 + r0) if (p0 + r0) > 0 else 0.0

def manual_macro_f1_score(y_true, y_pred):
    f1_1 = manual_f1_score(y_true, y_pred)
    f1_0 = manual_f1_score_class0(y_true, y_pred)
    return (f1_1 + f1_0) / 2.0

# Funkcija za ručno kreiranje sekvenci (3D ulaz)
def create_sequences(X, y, lookback=10):
    X_arr = np.array(X)
    y_arr = np.array(y)
    Xs, ys = [], []
    for i in range(len(X_arr) - lookback):
        Xs.append(X_arr[i:(i + lookback)])
        ys.append(y_arr[i + lookback])
    return np.array(Xs), np.array(ys)

# Optimizacija praga na temelju Macro F1 metrike
def optimize_threshold(y_true, y_proba):
    best_thresh = 0.50
    best_macro = 0.0
    for thresh in np.arange(0.35, 0.65, 0.01):
        preds = (y_proba >= thresh).astype(int)
        macro = manual_macro_f1_score(y_true, preds)
        if macro > best_macro:
            best_macro = macro
            best_thresh = thresh
    return float(best_thresh)

lstm_results = []
lstm_models = {}
lstm_predictions = {}
lstm_thresholds = {}

for ticker in TICKERS:
    print(f"Obučavanje LSTM-a za {ticker}...")
    X_tr, y_tr, X_vl, y_vl, X_te, y_te = splits[ticker]

    # Obavezno skaliranje pomoću ručnog skalera
    scaler = ManualStandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_vl_scaled = scaler.transform(X_vl)

    # Kreiranje sekvenci za vremenske serije
    X_tr_lstm, y_tr_lstm = create_sequences(X_tr_scaled, y_tr, LOOKBACK)
    X_vl_lstm, y_vl_lstm = create_sequences(X_vl_scaled, y_vl, LOOKBACK)

    # Težine klasa za izbalansirano treniranje
    neg_count = np.sum(y_tr_lstm == 0)
    pos_count = np.sum(y_tr_lstm == 1)
    total_count = len(y_tr_lstm)
    w0 = total_count / (2.0 * neg_count)
    w1 = total_count / (2.0 * pos_count)
    class_weight_dict = {0: float(w0), 1: float(w1)}

    # Definicija robusne LSTM arhitekture s tanh aktivacijom i L2 regularizacijom
    model = Sequential([
        LSTM(32, input_shape=(LOOKBACK, X_tr_scaled.shape[1]), activation="tanh", kernel_regularizer=l2(0.005)),
        Dropout(0.3),
        Dense(16, activation="relu", kernel_regularizer=l2(0.005)),
        Dropout(0.2),
        Dense(1, activation="sigmoid"),
    ])

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    
    # Treniranje s ranim zaustavljanjem na temelju validacijskog gubitka
    from tensorflow.keras.callbacks import EarlyStopping
    early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    
    model.fit(
        X_tr_lstm,
        y_tr_lstm,
        epochs=20,
        batch_size=64,
        validation_data=(X_vl_lstm, y_vl_lstm),
        class_weight=class_weight_dict,
        callbacks=[early_stopping],
        verbose=0,
    )

    # Predviđanje vjerojatnosti na validacijskom skupu
    y_proba = model.predict(X_vl_lstm, verbose=0).flatten()
    
    # Optimizacija praga
    t_lstm = optimize_threshold(y_vl_lstm, y_proba)
    lstm_thresholds[ticker] = t_lstm
    
    # Predikcija na temelju optimiziranog praga
    y_pred = (y_proba >= t_lstm).astype(int)

    # Metrike računamo na y_vl_lstm i y_pred pomoću naših ručnih funkcija
    lstm_results.append({
        "Ticker": ticker,
        "Model": "LSTM",
        "Accuracy": manual_accuracy(y_vl_lstm, y_pred),
        "Precision": manual_precision(y_vl_lstm, y_pred),
        "Recall": manual_recall(y_vl_lstm, y_pred),
        "F1": manual_f1_score(y_vl_lstm, y_pred),
    })

    lstm_models[ticker] = (model, scaler)
    # Spremamo predikciju, vjerojatnost i stvarni y_vl_lstm za crtanje matrice zabune
    lstm_predictions[ticker] = (y_pred, y_proba, y_vl_lstm)
    print(f"   Uspješno obučena LSTM mreža za {ticker} | Prag: {t_lstm:.2f} | Validation F1 = {lstm_results[-1]['F1']:.4f}")

lstm_df = pd.DataFrame(lstm_results).set_index("Ticker")
display(lstm_df.round(4))


### 14.1 Matrice zabune za LSTM modele

Crtamo matrice zabune za LSTM model kako bismo vidjeli koliki je omjer ispravno i neispravno klasificiranih dana na validacijskom skupu.

In [ ]:
n_cols = min(2, len(TICKERS))
n_rows = int(np.ceil(len(TICKERS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 4.5 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, ticker in zip(axes, TICKERS):
    # Za sekvencijalni LSTM, koristimo stvarni y_val_lstm koji je spremljen u lstm_predictions
    y_pred, _, y_val_lstm = lstm_predictions[ticker]
    
    # Ručna matrica zabune
    cm = manual_confusion_matrix(y_val_lstm, y_pred)
    cm_pct = cm / cm.sum() * 100
    annot = np.array([
        [f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)" for j in range(2)]
        for i in range(2)
    ])

    sns.heatmap(
        cm,
        annot=annot,
        fmt="",
        cmap="Purples",
        ax=ax,
        xticklabels=["Pad", "Rast"],
        yticklabels=["Pad", "Rast"],
        cbar=False,
        annot_kws={"fontsize": 10},
    )
    ax.set_title(f"LSTM - {ticker} Matrica zabune")
    ax.set_xlabel("Predviđeno")
    ax.set_ylabel("Stvarno")

for ax in axes[len(TICKERS):]:
    fig.delaxes(ax)

plt.suptitle("Matrice zabune za LSTM neuronske mreže (Validation set)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 15. Spremanje modela za web aplikaciju

Za integraciju rezultata u Flask web poslužitelj, serijaliziramo (spremamo) sve obučene modele. Koristimo:
- `joblib` za klasične modele (Logistička regresija, Random Forest, XGBoost) i pripadajuće skalere.
- Keras `.keras` format za LSTM neuronske mreže.

In [ ]:
print("Spremanje modela i skalera za dionice obučene u bilježnici...")
import json

for ticker in TICKERS:
    # Spremanje klasičnih modela
    for name, (model, scaler) in all_models[ticker].items():
        model_filename = name.replace(' ', '_')
        joblib.dump(model, OUTPUT_DIR / f'{ticker}_{model_filename}.joblib')
        if scaler is not None:
            joblib.dump(scaler, OUTPUT_DIR / f'{ticker}_scaler.joblib')
            
    # Spremanje LSTM modela
    lstm_model, lstm_scaler = lstm_models[ticker]
    lstm_model.save(OUTPUT_DIR / f'{ticker}_LSTM.keras')
    joblib.dump(lstm_scaler, OUTPUT_DIR / f'{ticker}_scaler.joblib')

# Spremanje optimalnih pragova u thresholds.json
thresholds_path = OUTPUT_DIR / 'thresholds.json'
thresh_dict = {}
if thresholds_path.exists():
    try:
        with open(thresholds_path, 'r') as f:
            thresh_dict = json.load(f)
    except:
        pass

for ticker in TICKERS:
    if ticker not in thresh_dict:
        thresh_dict[ticker] = {}
    for name in all_models[ticker].keys():
        thresh_dict[ticker][name] = 0.50
    thresh_dict[ticker]['LSTM'] = float(lstm_thresholds[ticker])

with open(thresholds_path, 'w') as f:
    json.dump(thresh_dict, f, indent=4)
    
print(f"\nSvi modeli i optimalni pragovi uspješno spremljeni u mapu: {OUTPUT_DIR.resolve()}")


## 16. Zaključak

U ovom radu uspješno je razvijen sustav za klasifikaciju kratkoročnog kretanja cijena dionica primjenom metoda strojnog i dubokog učenja. Tijekom rada proveden je cjelokupan proces znanosti o podacima:

1. **Prikupljanje podataka**: Dohvaćeni su povijesni podaci za 54 dionice te ključni makroekonomski indikatori (S&P 500, VIX) preko yfinance sučelja.
2. **Inženjerstvo značajki**: Izgrađeno je 25 tehničkih i makrosustavnih indikatora za opis tržišnog konteksta dionica na dnevnoj razini.
3. **Evaluacijska metodologija**: Implementirana je stroga kronološka podjela na train, validation i test skupove uz TimeSeriesSplit kako bi se spriječilo curenje podataka.
4. **Ručna implementacija**: Izgrađeni su ručni StandardScaler i funkcije za evaluacijske metrike (točnost, preciznost, odziv, F1-score i konfuzijska matrica), čime smo dokazali poznavanje matematike u pozadini modela.
5. **Usporedba modela**: Testirana su 4 algoritma na testnom skupu (2024. godina). Usporedba s pasivnom strategijom Buy & Hold otkrila je da aktivne ML strategije mogu smanjiti rizik (povlačenjem kapitala kada se predviđa pad) i u određenim uvjetima ostvariti alfu (nadmašivanje pasivnog tržišta).
6. **Integracija u aplikaciju**: Obučeni modeli i skaler uspješno su spremljeni i povezani s Flask poslužiteljem i TradingView interaktivnim grafičkim sučeljem.